# 17. 팀원 방식 vs 내 방식 — 같은 두 틀에서 맞대결

| 틀 | 데이터 | 단위 | 핵심 지표 |
|---|---|---|---|
| **① 팀원 틀** | 공식 2차만 (CN7 10/16·10/20, RG3 10/21~23) | 행 (LH/RH) | AP·무작위 범위, 파일 전체 상위 k%, **예산 F1(5%)**, **정보 상한 대비** |
| **② 내 틀** | labeled 전체 (2차+추가, 초기허용 제외) | 샷 | AP, **Run별 hit@10%**, 유형별, 예산 F1 |

**방법**
- 팀원(JIWOO 07·08·10 재현): `T_prev10` 직전 10샷 / `T_global` 파일 전체(미래 포함, 참고) / `T_past10` 과거 기준선 K10 / `T_phys300` 물리 관계 5개
- 나: `M_knn_free`, `M_maha_free` (reference 이상탐지, 0 모드 제거) / `M_rule_phys` (학습 없는 물리 규칙) / (있으면) `M_E2_phys` (13번, 라벨 사용 — 참고)

**판정 기준**: 두 틀 모두에서 위에 있는 방법 = 최종 후보. 한 틀에서만 좋으면 그 틀의 착시(Run 혼동, 쌍둥이 행 등)를 의심

In [1]:
%load_ext autoreload
%autoreload 2
import os, json
import numpy as np, pandas as pd
from compare_frames import (load_all_shots, load_unlabeled_normal, official_rows, t_prev, t_global, t_past, t_phys,
                            build_ref_nozero, my_scores, frame1_eval, frame2_oof, frame2_eval)
from model_phys import add_phys
from scope import apply_scope
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 40); pd.set_option('display.max_rows', 200)
OUT = './outputs/compare_frames'
os.makedirs(OUT, exist_ok=True)
PRODUCTS = ['cn7', 'rg3']
SCALES = json.load(open('./base/reference_scales.json', encoding='utf-8'))
REF_PERIOD = ('2020-09-01', '2020-11-30')

## 1. 점수 계산 (전 샷, 시간 흐름 그대로) — 팀원 방식은 파이썬 반복이라 몇 분 걸림

In [2]:
SHOTS, FEAT, SC, REFSC = {}, {}, {}, {}
for p in PRODUCTS:
    sh = load_all_shots(p)
    unl = load_unlabeled_normal(p)
    s = {}
    s['T_prev10'] = t_prev(sh, 10)
    s['T_past10'] = t_past(sh, unl, 10)
    s['T_phys300'] = t_phys(sh, unl, 300)
    s['T_global(미래포함)'] = t_global(sh)                       # ② 틀용 (labeled 전체 기준)
    m2 = (sh['split'] == '2차').values
    g2 = np.zeros(len(sh)); g2[m2] = t_global(sh[m2].reset_index(drop=True))
    s['T_global_2차(미래포함)'] = g2                              # ① 틀용 (팀원과 같이 2차 파일만 기준)
    SC[p] = {k: pd.Series(v, index=sh['TimeStamp'].values) for k, v in s.items()}
    SHOTS[p] = sh
    print(p, '팀원 방식 완료')

    # 내 방식: base features (전 샷) + 0 모드 제거 reference
    f = add_phys(pd.read_csv(f'./base/{p}_features.csv', parse_dates=['TimeStamp']))
    f['Reason'] = f['Reason'].fillna('정상')
    cache = f'./base/ref_{p}_features_nozero_{REF_PERIOD[0]}_{REF_PERIOD[1]}.csv'
    if os.path.exists(cache):
        ref = add_phys(pd.read_csv(cache, parse_dates=['TimeStamp']))
    else:
        ref = build_ref_nozero(p, REF_PERIOD, SCALES); ref.to_csv(cache, index=False, encoding='utf-8-sig')
    for k, (sf, sr) in my_scores(f, ref).items():
        SC[p][k] = pd.Series(sf, index=f['TimeStamp'].values)
        REFSC.setdefault(p, {})[k] = sr
    FEAT[p] = f
    print(p, '내 방식 완료')

# (선택) 13번 E2_phys — 있으면 추가 (run_id + 샷 순번으로 시각 연결)
P13 = './outputs/phys_model/oof.csv'
if os.path.exists(P13):
    o13 = pd.read_csv(P13)
    for p in PRODUCTS:
        o = o13[(o13['product'] == p) & (o13['method'] == 'E2_phys|과도제외')]
        key = FEAT[p].set_index(['run_id', 'shot_seq_in_run'])['TimeStamp']
        ts = [key.get((r, s)) for r, s in zip(o['run_id'], o['shot_seq'])]
        SC[p]['M_E2_phys(라벨사용)'] = pd.Series(o['score'].values, index=ts)
    print('13번 E2_phys 추가')

[cn7] 비라벨 28744샷 중 0 모드 18139 (63.1%) 제거


c:\anaconda\envs\kamp\Lib\site-packages\pandas\core\arraylike.py:402: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)


cn7 팀원 방식 완료
[cn7] reference: unlabeled 28,744샷 → 16,353샷, Run 10개 (2020-09-03 ~ 2020-10-19), 이상기록 제거 1634
[cn7] reference 0 모드 11174샷 제거
cn7 내 방식 완료
[rg3] 비라벨 29633샷 중 0 모드 13119 (44.3%) 제거


c:\anaconda\envs\kamp\Lib\site-packages\pandas\core\arraylike.py:402: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
c:\anaconda\envs\kamp\Lib\site-packages\pandas\core\arraylike.py:402: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
c:\anaconda\envs\kamp\Lib\site-packages\pandas\core\arraylike.py:402: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)


rg3 팀원 방식 완료
[rg3] reference: unlabeled 29,633샷 → 4,406샷, Run 7개 (2020-09-14 ~ 2020-10-22), 이상기록 제거 0
[rg3] reference 0 모드 47샷 제거
rg3 내 방식 완료
13번 E2_phys 추가


## 2. ① 팀원 틀 — 공식 2차, 행 단위

In [3]:
F1T = []
for p in PRODUCTS:
    rows = official_rows(p)
    sc = {k: v for k, v in SC[p].items() if k != 'T_global(미래포함)'}
    F1T.append(frame1_eval(p, rows, sc))
F1T = pd.concat(F1T, ignore_index=True).sort_values(['제품', 'F1@5%'], ascending=[True, False])
F1T.to_csv(f'{OUT}/frame1_official_rows.csv', index=False, encoding='utf-8-sig')
display(F1T.round(3))

,제품,방법,행,불량 행,AP,무작위 평균 대비,무작위 상단,유의,상위1%,상위5%,상위10%,F1@5%,TP@5%,F1@10%,TP@10%,정보 상한 F1,F1@5% / 상한
3,cn7,T_global_2차(미래포함),1211,17,0.224,11.833,0.044,O,6/17,8/17,13/17,0.205,8,0.188,13,0.756,0.271
1,cn7,T_past10,1211,17,0.205,10.793,0.044,O,2/17,7/17,10/17,0.179,7,0.145,10,0.756,0.238
2,cn7,T_phys300,1211,17,0.150,7.884,0.044,O,4/17,7/17,7/17,0.179,7,0.101,7,0.756,0.238
4,cn7,M_knn_free,1211,17,0.346,18.227,0.044,O,6/17,7/17,7/17,0.179,7,0.101,7,0.756,0.238
6,cn7,M_rule_phys,1211,17,0.161,8.470,0.044,O,6/17,7/17,8/17,0.179,7,0.116,8,0.756,0.238
0,cn7,T_prev10,1211,17,0.125,6.594,0.044,O,2/17,5/17,10/17,0.154,6,0.145,10,0.756,0.204
5,cn7,M_maha_free,1211,17,0.373,19.655,0.044,O,6/17,6/17,8/17,0.154,6,0.116,8,0.756,0.204
7,cn7,M_E2_phys(라벨사용),1211,17,0.245,12.903,0.044,O,6/17,6/17,7/17,0.154,6,0.101,7,0.756,0.204
8,rg3,T_prev10,1182,25,0.022,0.802,0.057,-,0/25,1/25,2/25,0.024,1,0.028,2,0.667,0.036
9,rg3,T_past10,1182,25,0.020,0.761,0.057,-,0/25,1/25,1/25,0.024,1,0.014,1,0.667,0.036


## 3. ② 내 틀 — labeled 전체, 샷 단위 (초기허용 제외)

In [4]:
O2 = []
for p in PRODUCTS:
    f = apply_scope(FEAT[p][FEAT[p]['split'] != '제외'], exclude_initial=True)
    sc = {k: v for k, v in SC[p].items() if k != 'T_global_2차(미래포함)'}
    O2.append(frame2_oof(p, f, sc, REFSC[p]))
O2 = pd.concat(O2, ignore_index=True)
F2T = frame2_eval(O2)
F2T.to_csv(f'{OUT}/frame2_allruns_shots.csv', index=False, encoding='utf-8-sig')
cols = ['product', 'method', 'n_def', 'AP', 'AP_lo', 'AP_hi', 'AP/base', 'hit@10%', '전체 상위10%',
        'F1@5%(전체)', 'AP_가스', 'hit10_가스', 'AP_미성형', 'hit10_미성형']
display(F2T[cols].sort_values(['product', 'AP'], ascending=[True, False]).round(3))

,product,method,n_def,AP,AP_lo,AP_hi,AP/base,hit@10%,전체 상위10%,F1@5%(전체),AP_가스,hit10_가스,AP_미성형,hit10_미성형
0,cn7,T_global(미래포함),15,0.235,0.130,0.495,30.866,7/15,13/15,0.211,0.130,4/11,0.290,3/4
1,cn7,M_maha_free,15,0.212,0.011,0.416,27.843,4/15,5/15,0.070,0.007,1/11,0.751,3/4
2,cn7,M_E2_phys(라벨사용),15,0.137,0.009,0.392,17.996,3/15,4/15,0.053,0.006,0/11,0.480,3/4
3,cn7,M_knn_free,15,0.129,0.010,0.354,17.027,4/15,4/15,0.070,0.007,1/11,0.442,3/4
4,cn7,M_rule_phys,15,0.090,0.011,0.310,11.847,5/15,5/15,0.070,0.008,2/11,0.288,3/4
5,cn7,T_phys300,15,0.073,0.013,0.265,9.667,7/15,5/15,0.070,0.009,4/11,0.228,3/4
6,cn7,T_past10,15,0.038,0.014,0.152,5.065,6/15,6/15,0.070,0.014,3/11,0.065,3/4
7,cn7,T_prev10,15,0.035,0.014,0.125,4.606,5/15,6/15,0.053,0.013,2/11,0.056,3/4
8,rg3,T_global(미래포함),32,0.118,0.054,0.226,2.316,4/32,7/32,0.159,0.106,2/22,0.043,2/10
9,rg3,M_rule_phys,32,0.085,0.058,0.181,1.668,4/32,5/32,0.063,0.072,3/22,0.027,1/10


## 4. 한 표로 — 두 틀 순위

In [5]:
a = F1T[['제품', '방법', 'AP', 'F1@5%', 'F1@5% / 상한', '상위10%']].rename(
    columns={'제품': 'product', '방법': 'method', 'AP': '①AP', 'F1@5%': '①F1@5%', 'F1@5% / 상한': '①상한대비', '상위10%': '①상위10%'})
a['method'] = a['method'].replace({'T_global_2차(미래포함)': 'T_global(미래포함)'})
b = F2T[['product', 'method', 'AP', 'hit@10%', 'hit10_미성형', 'hit10_가스']].rename(
    columns={'AP': '②AP', 'hit@10%': '②Run별hit10'})
BOTH = a.merge(b, on=['product', 'method'], how='outer')
BOTH['①순위'] = BOTH.groupby('product')['①AP'].rank(ascending=False)
BOTH['②순위'] = BOTH.groupby('product')['②AP'].rank(ascending=False)
BOTH = BOTH.sort_values(['product', '②순위'])
BOTH.to_csv(f'{OUT}/both_frames.csv', index=False, encoding='utf-8-sig')
display(BOTH.round(3))

,product,method,①AP,①F1@5%,①상한대비,①상위10%,②AP,②Run별hit10,hit10_미성형,hit10_가스,①순위,②순위
4,cn7,T_global(미래포함),0.224,0.205,0.271,13/17,0.235,7/15,3/4,4/11,4.0,1.0
2,cn7,M_maha_free,0.373,0.154,0.204,8/17,0.212,4/15,3/4,1/11,1.0,2.0
0,cn7,M_E2_phys(라벨사용),0.245,0.154,0.204,7/17,0.137,3/15,3/4,0/11,3.0,3.0
1,cn7,M_knn_free,0.346,0.179,0.238,7/17,0.129,4/15,3/4,1/11,2.0,4.0
3,cn7,M_rule_phys,0.161,0.179,0.238,8/17,0.090,5/15,3/4,2/11,6.0,5.0
6,cn7,T_phys300,0.150,0.179,0.238,7/17,0.073,7/15,3/4,4/11,7.0,6.0
5,cn7,T_past10,0.205,0.179,0.238,10/17,0.038,6/15,3/4,3/11,5.0,7.0
7,cn7,T_prev10,0.125,0.154,0.204,10/17,0.035,5/15,3/4,2/11,8.0,8.0
12,rg3,T_global(미래포함),0.019,0.000,0.000,1/25,0.118,4/32,2/10,2/22,7.0,1.0
11,rg3,M_rule_phys,0.034,0.024,0.036,3/25,0.085,4/32,1/10,3/22,1.0,2.0


## 해석 가이드
- **두 틀 모두 상위** → 최종 후보. 한 틀에서만 상위면 이유를 확인
  - ①에서만 높음: Run 2개(10/16 vs 10/20) 구분 효과일 가능성 (특히 `T_global`, 수준값 기반)
  - ②에서만 높음: 추가 기간 Run의 사건을 잡은 것 — 유형별 hit로 무엇을 잡았는지 확인
- **① 상한 대비**: 정보 상한(쌍둥이 행 때문에 넘을 수 없는 F1) 대비 몇 % — 보고서용
- **CN7 미성형 hit**는 대부분 3/4일 것 → 차이는 가스·오탐에서 남. 오탐이 적은 쪽(예산 F1)이 운영상 유리
- RG3는 전부 무작위 근처일 것 — 그게 결론 (방법 문제가 아니라 데이터 한계)
- 팀원 방식은 원값 21변수로 재현(팀원 24변수 중 상수·기록오류 3개 제외) → 팀원 노트북 수치와 소수점은 다를 수 있음